# H&M Fashion Recommender — FashionCLIP Image Embeddings

Encodes every product image into a 512-dim embedding using [FashionCLIP](https://huggingface.co/patrickjohncyh/fashion-clip), a CLIP model fine-tuned on fashion data. These embeddings power the image and text search.

**Why FashionCLIP over generic CLIP:** generic CLIP produced near-identical similarity scores across very different garments (everything clustered ~0.93–0.95), making ranking unreliable. FashionCLIP's fashion-tuned embeddings discriminate clearly by garment type and colour.

Steps: Setup → Load Model → Load Articles → Encode Images (batched, checkpointed) → Verify

## 1. Setup

Mount Drive and install dependencies. FashionCLIP is distributed only as a PyTorch model, so the encoding runs on `transformers` + `torch`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Load FashionCLIP

Loads the model and processor. Note: in this `transformers` version, `get_image_features` returns a pooled-output object rather than the projected vector, so embeddings are taken explicitly via `model.vision_model` + `model.visual_projection` to get the correct 512-dim CLIP-space vectors.

In [ ]:
!pip install -q transformers torch pillow

In [ ]:
import sys, os
sys.path.append('/content/drive/MyDrive/hm_fashion_project/src')
import torch
import numpy as np
import pandas as pd
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from paths import BASE_PROCESS, IMAGE_PATH

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("patrickjohncyh/fashion-clip").to(device)
processor = CLIPProcessor.from_pretrained("patrickjohncyh/fashion-clip")
model.eval()
print("fashion-clip loaded on", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

fashion-clip loaded on cpu


## 3. Load the Image-Having Article Set

Loads `articles_with_images.parquet` — the subset of articles that have a product image (71,664 items). Row order here defines the embedding row order, which is critical for mapping FAISS results back to articles later.

In [ ]:
df_final = pd.read_parquet(os.path.join(BASE_PROCESS, "articles/articles_with_images.parquet"))
df_final['article_id'] = df_final['article_id'].astype(str).str.zfill(10)
print(df_final.shape)

(71664, 12)


## 4. Encode Images

Encodes all images in batches. Run on free Colab, this faced session resets, Drive read throttling, and RAM limits — so it uses **checkpointing** (saving progress every N batches to `fclip_image_partial.npy`) to resume after interruptions, mixed-precision autocast for speed, and `num_workers=0` to avoid DataLoader memory crashes. Each image is encoded through the vision model and visual projection, then L2-normalized for cosine-similarity (inner-product) search.

In [ ]:
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import time

class ImgData(Dataset):
    def __init__(self, ids):
        self.ids = ids
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        aid = self.ids[i]
        path = os.path.join(IMAGE_PATH, aid[:3], f"{aid}.jpg")
        try:
            return Image.open(path).convert("RGB")
        except Exception:
            return Image.new("RGB", (224, 224), (255, 255, 255))

def collate(imgs):
    return processor(images=imgs, return_tensors="pt", padding=True)

ids = df_final['article_id'].tolist()
BATCH = 128
ckpt = os.path.join(BASE_PROCESS, "embeddings/fclip_image_partial.npy")

# load checkpoint, figure out where to resume
all_emb = []
done_idx = 0
if os.path.exists(ckpt):
    saved = np.load(ckpt)
    all_emb = [saved]
    done_idx = len(saved)
    print(f"resuming from {done_idx}")

# only build dataset from the REMAINING ids - no skip loop
remaining_ids = ids[done_idx:]
loader = DataLoader(ImgData(remaining_ids), batch_size=BATCH, collate_fn=collate,
                    num_workers=0, pin_memory=True)

for batch in tqdm(loader, total=len(loader)):
    batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            vision_out = model.vision_model(pixel_values=batch['pixel_values'])
            feat = model.visual_projection(vision_out.pooler_output)
            feat = feat / feat.norm(dim=-1, keepdim=True)
    all_emb.append(feat.cpu().numpy())
    np.save(ckpt, np.vstack(all_emb))   # save every batch now since net is unstable

image_emb = np.vstack(all_emb).astype('float32')
np.save(os.path.join(BASE_PROCESS, "embeddings/fclip_image_embeddings.npy"), image_emb)
print("done", image_emb.shape)

resuming from 67200


  0%|          | 0/35 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_6210/1866751183.py:42: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):
100%|██████████| 35/35 [39:49<00:00, 68.28s/it]


done (71664, 512)


## 5. Verify

Load the final saved embeddings and confirm the shape is (71664, 512).

In [ ]:
fclip = np.load(os.path.join(BASE_PROCESS, "embeddings/fclip_image_embeddings.npy"))
print(fclip.shape, fclip.dtype)

(71664, 512) float32
